# Instamart Support Copilot — Parts C & D

**Project notebook**

This notebook covers support-ticket triage and the policy/refund assistant. It uses the cleaned tables and customer segments produced in Notebook 1. Run **Part A & B first** so that `data/clean/`, `data/instamart.db`, and `outputs/customer_segments.csv` are available.

The implementation lives in the project's `src/` modules. This notebook runs experiments, checks outputs, and documents the results.

## 1. Set up the project and check prerequisites

In [1]:
from pathlib import Path
import sys
import subprocess
import json
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "generate_data.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "generate_data.py").exists():
    raise FileNotFoundError("Open this notebook from inside the project folder.")
sys.path.insert(0, str(ROOT))
print("Project folder:", ROOT)

required_paths = [
    ROOT / "data" / "clean" / "customers.csv",
    ROOT / "data" / "clean" / "orders.csv",
    ROOT / "data" / "instamart.db",
    ROOT / "outputs" / "customer_segments.csv",
]
for path in required_paths:
    print("OK" if path.exists() else "MISSING", path.relative_to(ROOT))
if not all(path.exists() for path in required_paths):
    raise FileNotFoundError("Run Notebook 1 (Part A & B) first.")

Project folder: c:\Users\asus\OneDrive\Desktop\with_notebooks_Instamart_Customer_Intelligence_Copilot\instamart-customer-intelligence-copilot
OK data\clean\customers.csv
OK data\clean\orders.csv
OK data\instamart.db
OK outputs\customer_segments.csv


## 2. Part C — protect and classify support tickets

The triage workflow handles empty and overly long tickets before model calls, masks email/phone/order identifiers, retries transient failures, validates the returned category, applies a token budget, and routes tickets using both category and customer segment.

In [2]:
from src.triage import redact, run_pipeline, ZERO, FEW

example_text = "Please check order ORD-123456 and contact kiru@example.com or 9876543210."
redacted_text, redaction_counts = redact(example_text)
print("Before:", example_text)
print("After: ", redacted_text)
print("Masked items:", redaction_counts)

Before: Please check order ORD-123456 and contact kiru@example.com or 9876543210.
After:  Please check order [ORDER_ID] and contact [EMAIL] or [PHONE].
Masked items: {'email': 1, 'phone': 1, 'order_id': 1}


### Run the zero-shot and few-shot experiments

In [3]:
raw_tickets = pd.read_csv(ROOT / "data" / "raw" / "tickets.csv", keep_default_na=False)
print("Tickets:", len(raw_tickets))
print("Empty ticket texts:", int(raw_tickets.text.astype(str).str.strip().eq("").sum()))
print("Longest ticket word counts:", raw_tickets.text.astype(str).str.split().str.len().sort_values(ascending=False).head().tolist())

zero_results, zero_summary = run_pipeline(prompt=ZERO, token_budget=100000)
few_results, few_summary = run_pipeline(prompt=FEW, token_budget=100000)

def score_triage(results, label):
    checked = results.merge(raw_tickets[["ticket_id", "true_category"]], on="ticket_id", how="left")
    classified = checked[checked.status.isin(["classified_json", "classified_keyword"])]
    accuracy = (classified.category == classified.true_category).mean() if len(classified) else 0.0
    return {
        "prompt": label,
        "classified_tickets": len(classified),
        "accuracy_on_classified": round(float(accuracy), 4),
        "tokens_used": int(zero_summary["tokens"] if label == "zero-shot" else few_summary["tokens"]),
        "status_counts": results.status.value_counts().to_dict(),
    }

display(pd.DataFrame([score_triage(zero_results, "zero-shot"), score_triage(few_results, "few-shot")]))
display(few_results.head(10))

Tickets: 164
Empty ticket texts: 4
Longest ticket word counts: [564, 564, 564, 564, 21]
retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=

,prompt,classified_tickets,accuracy_on_classified,tokens_used,status_counts
0,zero-shot,147,0.8571,7372,"{'classified_json': 135, 'classified_keyword':..."
1,few-shot,147,1.0000,12520,"{'classified_json': 135, 'classified_keyword':..."


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,Dormant,HIGH,security_team,84
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,80
2,T0095,C1087,classified_json,technical,Occasional,NORMAL,support_queue,82
3,T0159,C1172,skipped_empty,,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,Occasional,HIGH,security_team,84
5,T0124,C0139,classified_json,fraud,Frequent,HIGH,security_team,84
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,76
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,79
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,80
9,T0002,C0881,classified_json,delivery,Occasional,NORMAL,support_queue,78


### Check the token-budget guardrail

In [4]:
budget_results, budget_summary = run_pipeline(prompt=ZERO, token_budget=6000)
print("Token budget:", 6000)
print("Tokens used:", budget_summary["tokens"])
print("Budget-exhausted tickets:", int((budget_results.status == "budget_exhausted").sum()))
print("Saved:", ROOT / "outputs" / "triage_budget_demo.csv")

final_triage_path = ROOT / "outputs" / "triage_results.csv"
print("Final few-shot output exists:", final_triage_path.exists())
if final_triage_path.exists():
    display(pd.read_csv(final_triage_path).head(10))

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
Token budget: 6000
Tokens used: 6009
Budget-exhausted tickets: 29
Saved: c:\Users\asus\OneDrive\Desktop\with_notebooks_Instamart_Customer_Intelligence_Copilot\instamart-customer-intelligence-copilot\outputs\triage_budget_demo.csv
Final few-shot output exists: True


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,Dormant,HIGH,security_team,84
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,80
2,T0095,C1087,classified_json,technical,Occasional,NORMAL,support_queue,82
3,T0159,C1172,skipped_empty,NaN,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,Occasional,HIGH,security_team,84
5,T0124,C0139,classified_json,fraud,Frequent,HIGH,security_team,84
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,76
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,79
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,80
9,T0002,C0881,classified_json,delivery,Occasional,NORMAL,support_queue,78


### Routing checks

In [5]:
routing_summary = few_results.groupby(["route", "priority", "status"], dropna=False).size().rename("tickets").reset_index()
display(routing_summary.sort_values(["route", "status"]))
print("Redaction counts from full few-shot run:", few_summary["redactions"])
print("Run summary:", few_summary["status_counts"])

,route,priority,status,tickets
0,human_review,NORMAL,flagged_invalid_output,9
1,human_review,NORMAL,flagged_long,4
2,human_review,NORMAL,skipped_empty,4
3,premium_desk,HIGH,classified_json,10
4,premium_desk,HIGH,classified_keyword,1
5,security_team,HIGH,classified_json,31
6,security_team,HIGH,classified_keyword,3
7,support_queue,NORMAL,classified_json,94
8,support_queue,NORMAL,classified_keyword,8


Redaction counts from full few-shot run: {'email': 55, 'phone': 91, 'order_id': 132}
Run summary: {'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9, 'flagged_long': 4, 'skipped_empty': 4}


## 3. Part D — retrieve policy passages and answer questions

In [6]:
from src.policy_rag import PolicyRAG, evaluate

rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())

sample_question = "How long does a failed payment refund take if money was deducted?"
sample_answer = rag.answer(sample_question)
print("Question:", sample_question)
print("Answer:", sample_answer["answer"])
print("Sources:", sample_answer["sources"])
display(pd.DataFrame(sample_answer["retrieved"])[["source", "score", "text"]])

Policy chunks: 15


,source,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ..."
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours...."


Question: How long does a failed payment refund take if money was deducted?
Answer: If a payment fails but money is deducted, the amount is returned automatically
 
money is returned.
Sources: ['payment_policy.txt', 'refund_policy.txt']


,source,score,text
0,payment_policy.txt,0.314612,"Payment Policy. Customers can pay using UPI, d..."
1,refund_policy.txt,0.240784,money is returned. Approved refunds are credit...
2,refund_policy.txt,0.205463,Refund Policy. A refund can be requested only ...


### Evaluate retrieval with and without chunk overlap

In [7]:
evaluation_rows = []
for overlap in (10, 0):
    index = PolicyRAG(chunk_size=40, overlap=overlap)
    result = evaluate(index)
    answerable = result[result.expected_doc != "NONE"]
    unanswerable = result[result.expected_doc == "NONE"]
    evaluation_rows.append({
        "overlap": overlap,
        "chunks": len(index.chunks),
        "expected_document_retrieved": f"{int(answerable.retrieved_doc.sum())}/{len(answerable)}",
        "expected_fact_found": f"{int(answerable.fact_found.sum())}/{len(answerable)}",
        "unanswerable_fallbacks_correct": f"{int(unanswerable.fallback.sum())}/{len(unanswerable)}",
    })
display(pd.DataFrame(evaluation_rows))

,overlap,chunks,expected_document_retrieved,expected_fact_found,unanswerable_fallbacks_correct
0,10,15,12/12,12/12,2/2
1,0,13,12/12,10/12,2/2


## 4. Part D — run the refund-support agent scenarios

The agent uses tools to look up orders, retrieve policy passages, calculate a refund, and record an issue/escalation action. The run script also checks safe stopping, token/time fallbacks, and checkpoint/resume behaviour.

In [8]:
scenario_file = ROOT / "data" / "raw" / "agent_scenarios.csv"
scenarios = pd.read_csv(scenario_file, keep_default_na=False)
display(scenarios)

,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00
5,S6,C0001,Refund for order ORD-999999 please.,NOT_FOUND,0.00
6,S7,C0007,[LOOP] Refund order ORD-100048.,FALLBACK,0.00


In [9]:
# This script runs all seven required scenarios and the fallback/resume demonstrations.
scenario_run = subprocess.run(
    [sys.executable, str(ROOT / "run_agent_scenarios.py")],
    cwd=ROOT, check=True, capture_output=True, text=True
)
print(scenario_run.stdout)
if scenario_run.stderr.strip():
    print("Warnings / diagnostics:")
    print(scenario_run.stderr)

SCENARIOS
S1 REFUND 128.25 5 882 expected REFUND 128.25
S2 REFUND 135.0 5 882 expected REFUND 135.0
S3 ESCALATE 1999.0 5 869 expected ESCALATE 1999.0
S4 DENIED 0.0 4 593 expected DENIED 0.0
S5 DENIED 0.0 3 328 expected DENIED 0.0
S6 NOT_FOUND 0.0 2 66 expected NOT_FOUND 0.0
S7 FALLBACK_MAX_ITERATIONS 0.0 6 2982 expected FALLBACK 0.0
TOKEN_DEMO FALLBACK_TOKEN_LIMIT 3 345
TIME_DEMO FALLBACK_TIMEOUT 0 0
RESUME REFUND REFUND: INR 128.25 issued. ['get_order', 'search_policy', 'calculate_refund', 'issue_refund'] 5 882



### Review scenario results and safety demonstrations

In [10]:
agent_runs_path = ROOT / "outputs" / "agent_runs.jsonl"
if agent_runs_path.exists():
    agent_records = [json.loads(line) for line in agent_runs_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    scenario_summary = pd.DataFrame([{
        "scenario_id": row.get("scenario_id"),
        "status": row.get("status"),
        "refund_amount_inr": row.get("refund_amount", 0.0),
        "model_calls": row.get("llm_calls"),
        "tokens_used": row.get("tokens_used"),
        "expected_outcome": row.get("expected_outcome"),
    } for row in agent_records])
    display(scenario_summary)

print("Refund ledger:", (ROOT / "outputs" / "refund_ledger.jsonl").exists())
print("Resume checkpoint:", (ROOT / "state" / "resume-S1.json").exists())

,scenario_id,status,refund_amount_inr,model_calls,tokens_used,expected_outcome
0,S1,REFUND,128.25,5,882,REFUND
1,S2,REFUND,135.00,5,882,REFUND
2,S3,ESCALATE,1999.00,5,869,ESCALATE
3,S4,DENIED,0.00,4,593,DENIED
4,S5,DENIED,0.00,3,328,DENIED
5,S6,NOT_FOUND,0.00,2,66,NOT_FOUND
6,S7,FALLBACK_MAX_ITERATIONS,0.00,6,2982,FALLBACK


Refund ledger: True
Resume checkpoint: True


## 5. Final output checklist

In [11]:
deliverables = [
    "outputs/triage_results.csv",
    "outputs/triage_budget_demo.csv",
    "outputs/policy_eval_summary.json",
    "outputs/agent_runs.jsonl",
    "outputs/refund_ledger.jsonl",
    "state/resume-S1.json",
]
for relative_path in deliverables:
    path = ROOT / relative_path
    print(f"{'OK' if path.exists() else 'MISSING'}  {relative_path}")

OK  outputs/triage_results.csv
OK  outputs/triage_budget_demo.csv
OK  outputs/policy_eval_summary.json
OK  outputs/agent_runs.jsonl
OK  outputs/refund_ledger.jsonl
OK  state/resume-S1.json


## Part C & D — short conclusion

The triage pipeline combines privacy masking, input/output checks, retries and budget-aware routing. The policy assistant grounds its answer in retrieved policy passages and can fall back when the documents do not contain an answer. The refund agent demonstrates tool-based decisions with bounded execution, a human handoff path, and checkpoint/resume support. The CSV, JSONL and checkpoint files above are the evidence outputs for these experiments.